# HumanEval/MBPP retrieval: does QAChat find the right solution?

[CodeRAG-Bench](https://github.com/code-rag-bench/code-rag-bench) is a published benchmark for retrieval-augmented code generation; this notebook runs its evaluation protocol against QAChat's own real retrieval pipeline.

**The Problem.** Given a coding problem's own description, can QAChat's real retrieval pipeline correctly pick out that problem's reference solution from a corpus that also contains every other problem's solution?

**The Data.** Two of CodeRAG-Bench's canonical tasks — the simplest, most self-contained in its suite: HumanEval (164 problems) and MBPP (500 problems, test split only). Each is a document-to-code retrieval task in BEIR format: one query per problem (its prompt or description), one corpus document per problem (that problem's own canonical solution), and a relevance judgment marking exactly one corpus document as correct for each query — retrieval succeeds when that one solution ranks above the 163 (HumanEval) or 499 (MBPP) unrelated ones sitting next to it.

**The Metrics.** Standard information-retrieval ranking metrics, computed the same way BEIR and CodeRAG-Bench do (via `pytrec_eval`): NDCG, MRR, Recall, and Precision, each at several cutoffs (@1, @3, @5, @10, @100). 

**What's different from the original benchmark.** The tasks, corpus, queries, and relevance judgments are unmodified from CodeRAG-Bench's own data. The retrieval is QAChat's real pipeline: real chunking (`chunk_source_units`) instead of treating each solution as one indivisible document, and hybrid fusion (semantic + lexical + code-graph search through `hybrid_search`) instead of a single retriever. Because QAChat's chunker can split one corpus document into several chunks, the original document-level relevance judgments no longer line up directly with what's in the index, so each document's relevance is propagated to every chunk built from it before scoring — a hit on any fragment of the right solution still counts as finding it. This isn't a neutral fix, though: propagating relevance to several chunks instead of one changes what some of the metrics are rewarding, so the notebook also runs a whole-document baseline that matches CodeRAG-Bench's own un-chunked corpus exactly, to give a number that's directly comparable to the published benchmark.

## 1. Prepare QAChat

Locate the QAChat repo root, put it on the import path, and run QAChat's notebook setup.

In [1]:
from pathlib import Path

root = Path.cwd()
while not (root / "qachat").is_dir() and root != root.parent:
    root = root.parent

print(f"QAChat root: {root}")

QAChat root: /Users/cmanzo/Documents/GitHub/QAChat


In [2]:
import warnings

# A few MBPP canonical solutions use regex patterns as plain strings, e.g.
# '\w+\S*$' instead of r'\w+\S*$' -- that's MBPP's own data, unmodified
# per the intro above, not anything QAChat or this notebook writes. QAChat's
# ast.parse(source) (sources_py.py) surfaces Python's own SyntaxWarning for
# it while extracting functions/classes from that corpus text; ast.parse's
# default filename ("<unknown>") is why the warning doesn't show a real
# path. Harmless -- silenced here so it doesn't clutter later output.
warnings.filterwarnings("ignore", category=SyntaxWarning, message="invalid escape sequence")


In [3]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [4]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

QAChat environment ready: /Users/cmanzo/Documents/GitHub/QAChat/.venv/bin/python


## 2. Benchmark-only dependencies

Check that Python packages needed for the benchmark are installed.

In [5]:
try:
    import datasets
    import pytrec_eval
except ImportError as error:
    raise RuntimeError(
        "Benchmark dependencies are missing. From the QAChat root, run:\n"
        "    uv sync --extra mlx --extra torch --extra bench\n"
        "(always pass every extra together -- a bare '--extra bench' or\n"
        "'--extra mlx --extra torch' will each uninstall what the other\n"
        "installed) then select this notebook's kernel again."
    ) from error

print(f"datasets {datasets.__version__}")

/Users/cmanzo/Documents/GitHub/QAChat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


datasets 5.0.1


## 3. Build the HumanEval and MBPP retrieval tasks

Load HumanEval and MBPP from Hugging Face and reshape each into a 
`BenchmarkTask` (query, corpus, qrels) matching CodeRAG-Bench's own BEIR 
format.

In [6]:
from benchmarks.common import BenchmarkTask


def build_humaneval_task() -> BenchmarkTask:
    """Build the HumanEval document-to-code retrieval task.

    Mirrors CodeRAG-Bench's ``retrieval/create/humaneval.py``: the query is
    a problem's prompt, and the matching corpus document is that same
    prompt followed by its canonical solution.
    """

    rows = datasets.load_dataset("openai/openai_humaneval")["test"]

    corpus: dict[str, dict[str, str]] = {}
    queries: dict[str, str] = {}
    qrels: dict[str, dict[str, int]] = {}

    for row in rows:
        doc_id = f"{row['task_id']}_doc"
        code_id = f"{row['task_id']}_code"
        queries[doc_id] = row["prompt"]
        corpus[code_id] = {
            "title": row["entry_point"],
            "text": row["prompt"] + "\n" + row["canonical_solution"],
        }
        qrels[doc_id] = {code_id: 1}

    return BenchmarkTask("humaneval", corpus, queries, qrels)


humaneval_task = build_humaneval_task()
print(
    f"{humaneval_task.name}: {len(humaneval_task.queries)} queries, "
    f"{len(humaneval_task.corpus)} corpus documents"
)

humaneval: 164 queries, 164 corpus documents


In [7]:
def _mbpp_function_name(code_text: str) -> str:
    """Return the function name declared in one MBPP solution."""

    for line in code_text.split("\n"):
        stripped = line.lstrip()
        if stripped.startswith("def "):
            return stripped[4:].split("(")[0]
    return ""


def build_mbpp_task() -> BenchmarkTask:
    """Build the MBPP document-to-code retrieval task.

    Mirrors CodeRAG-Bench's ``retrieval/create/mbpp.py``. Only the test
    split is used here.
    """

    rows = datasets.load_dataset("google-research-datasets/mbpp")["test"]

    corpus: dict[str, dict[str, str]] = {}
    queries: dict[str, str] = {}
    qrels: dict[str, dict[str, int]] = {}

    for row in rows:
        doc_id = f"{row['task_id']}_doc"
        code_id = f"{row['task_id']}_code"
        queries[doc_id] = row["text"]
        corpus[code_id] = {
            "title": _mbpp_function_name(row["code"]),
            "text": "# " + row["text"] + "\n" + row["code"],
        }
        qrels[doc_id] = {code_id: 1}

    return BenchmarkTask("mbpp", corpus, queries, qrels)


mbpp_task = build_mbpp_task()
print(
    f"{mbpp_task.name}: {len(mbpp_task.queries)} queries, "
    f"{len(mbpp_task.corpus)} corpus documents"
)

mbpp: 500 queries, 500 corpus documents


In [8]:
tasks = (humaneval_task, mbpp_task)

## 4. Load the embedding model

Loads QAChat's embedding model, picking a backend (MLX or PyTorch) 
automatically -- needed early because `chunk_source_units` (next section) uses 
its real token counter for chunking.

In [9]:
BACKEND = "auto"
ALLOW_DOWNLOAD = True
MODEL_BY_BACKEND = {
    "mlx": "mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ",
    "torch": "Qwen/Qwen3-Embedding-0.6B",
}

In [10]:
from qachat import choose_backend

selected_backend = choose_backend(BACKEND)
MODEL_ID = MODEL_BY_BACKEND[selected_backend]

print(f"Backend: {selected_backend}")
print(f"Model: {MODEL_ID}")

Backend: mlx
Model: mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ


In [11]:
from qachat import EmbeddingSession

embedding_session = EmbeddingSession(
    MODEL_ID,
    backend=selected_backend,
    allow_download=ALLOW_DOWNLOAD,
)

print(f"Loaded on: {embedding_session.device}")

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 2912.09it/s]


Loaded on: metal


## 5. Turn the corpus into a real QAChat project

Write each corpus document to disk as its own `.py` file, then open that folder
as a real `QAProject` and run it through QAChat's actual chunking pipeline 
(`chunk_source_units`) -- exercising the same code path QAChat uses on a real 
repository.

In [12]:
def _doc_filename(doc_id: str) -> str:
    """Turn one corpus id into a safe, unique .py filename."""

    return doc_id.replace("/", "__") + ".py"


from benchmarks.common import corpus_already_written

project_root = root / "benchmarks" / "sandbox" / "humaneval_mbpp_benchmark"
profile_path = project_root / "profile.ini"
if not profile_path.exists():
    raise RuntimeError(
        f"Missing project profile: {profile_path}. This notebook expects "
        "the profile checked into the repository at that path."
    )

repository_dir = project_root / "repository"
path_to_doc_id: dict[str, str] = {
    f"{task.name}/{_doc_filename(doc_id)}": doc_id
    for task in tasks
    for doc_id in task.corpus
}

if corpus_already_written(repository_dir):
    print(f"Corpus already present under {repository_dir}, skipping write.")
else:
    for task in tasks:
        task_dir = repository_dir / task.name
        task_dir.mkdir(parents=True, exist_ok=True)
        for doc_id, doc in task.corpus.items():
            file_path = task_dir / _doc_filename(doc_id)
            file_path.write_text(doc["text"])
    print(f"Wrote {len(path_to_doc_id)} corpus documents under {repository_dir}")

Corpus already present under /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/humaneval_mbpp_benchmark/repository, skipping write.


In [13]:
from qachat.project import open_project

project = open_project(project_root)
profile = project.profile

print(f"Project source: {project.source}")
print(
    f"Chunking: target={profile.target_tokens} max={profile.max_tokens} "
    f"overlap={profile.overlap_tokens}"
)

Project source: /Users/cmanzo/Documents/GitHub/QAChat/benchmarks/sandbox/humaneval_mbpp_benchmark/repository
Chunking: target=300 max=400 overlap=40


In [14]:
from qachat.chunking import Chunk, chunk_source_units
from qachat.repository import scan_repository
from qachat.sources import extract_source_units, read_repository_files

repository_report = scan_repository(
    project.source,
    extensions=profile.extensions,
    max_file_bytes=profile.max_file_bytes,
)
source_files = read_repository_files(repository_report)
source_units = extract_source_units(source_files)
corpus_chunks = chunk_source_units(
    source_units,
    count_tokens=embedding_session.count_tokens,
    target_tokens=profile.target_tokens,
    max_tokens=profile.max_tokens,
    overlap_tokens=profile.overlap_tokens,
)

print(f"{len(source_files)} files -> {len(source_units)} source units -> {len(corpus_chunks)} chunks")

664 files -> 1217 source units -> 1223 chunks


## 6. Propagate document-level qrels to chunk level

Since real chunking can split one document into several chunks, give every 
resulting chunk its source document's relevance -- so a hit on any fragment of
the right solution still counts as finding it.


In [15]:
chunks_by_doc_id: dict[str, list[Chunk]] = {}
for chunk in corpus_chunks:
    doc_id = path_to_doc_id.get(str(chunk.path))
    if doc_id is None:
        continue
    chunks_by_doc_id.setdefault(doc_id, []).append(chunk)

split_counts: dict[str, int] = {}
for task in tasks:
    doc_ids = list(task.corpus)
    counts = [len(chunks_by_doc_id.get(doc_id, [])) for doc_id in doc_ids]
    split = sum(1 for count in counts if count > 1)
    empty = sum(1 for count in counts if count == 0)
    split_counts[task.name] = split
    print(
        f"{task.name}: {len(doc_ids)} documents, {split} split into >1 chunk, "
        f"{empty} produced 0 chunks (max chunks for one document: "
        f"{max(counts) if counts else 0})"
    )


def propagate_qrels(task: BenchmarkTask) -> dict[str, dict[str, int]]:
    """Give every chunk of a relevant document that document's relevance."""

    qrels_chunks: dict[str, dict[str, int]] = {}
    for query_id, doc_scores in task.qrels.items():
        chunk_scores: dict[str, int] = {}
        for doc_id, relevance in doc_scores.items():
            for chunk in chunks_by_doc_id.get(doc_id, ()):
                chunk_scores[chunk.identifier] = relevance
        qrels_chunks[query_id] = chunk_scores
    return qrels_chunks


qrels_chunks: dict[str, dict[str, dict[str, int]]] = {
    task.name: propagate_qrels(task) for task in tasks
}

missing = {
    task.name: sum(1 for scores in qrels_chunks[task.name].values() if not scores)
    for task in tasks
}
print(f"Queries left with zero relevant chunks after propagation: {missing}")


humaneval: 164 documents, 31 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 3)
mbpp: 500 documents, 500 split into >1 chunk, 0 produced 0 chunks (max chunks for one document: 5)
Queries left with zero relevant chunks after propagation: {'humaneval': 0, 'mbpp': 0}


### Verify: every chunk of a gold document is actually covered

Check that the propagation above did what it claims. For every query, every 
chunk belonging to that query's gold document should now carry the document's 
relevance in `qrels_chunks`, whatever rank that chunk comes back at.

In [16]:
mismatches = []

for task in tasks:
    for query_id, doc_scores in task.qrels.items():
        for doc_id, relevance in doc_scores.items():
            expected_chunk_ids = {chunk.identifier for chunk in chunks_by_doc_id.get(doc_id, ())}
            actual_chunk_ids = {
                chunk_id
                for chunk_id, score in qrels_chunks[task.name][query_id].items()
                if score == relevance
            }
            if expected_chunk_ids != actual_chunk_ids:
                mismatches.append((task.name, query_id, doc_id, expected_chunk_ids, actual_chunk_ids))

if mismatches:
    raise AssertionError(
        f"{len(mismatches)} document(s) have chunks missing from qrels_chunks "
        f"after propagation -- first: {mismatches[0]}"
    )

print("Every chunk of every gold document is present in qrels_chunks with the document's relevance.")

Every chunk of every gold document is present in qrels_chunks with the document's relevance.


## 7. Build a whole-document baseline (no splitting)

Build one unsplit chunk per corpus document -- the same granularity 
CodeRAG-Bench's own `qrels` already use, so no propagation is needed and this 
is directly comparable to the paper's own results.

In [17]:
def build_whole_document_chunks(task: BenchmarkTask) -> list[Chunk]:
    """Build one whole, unsplit chunk per corpus document."""

    chunks = []
    for doc_id, doc in task.corpus.items():
        text = doc["text"]
        chunks.append(
            Chunk(
                identifier=doc_id,
                path=Path(task.name) / _doc_filename(doc_id),
                text=text,
                start_line=1,
                end_line=text.count("\n") + 1,
                kind="whole_document",
                symbol=doc.get("title") or None,
                section=None,
            )
        )
    return chunks


whole_document_chunks: dict[str, list[Chunk]] = {
    task.name: build_whole_document_chunks(task) for task in tasks
}
for task in tasks:
    print(f"{task.name}: {len(whole_document_chunks[task.name])} whole-document chunks")

humaneval: 164 whole-document chunks
mbpp: 500 whole-document chunks


## 8. Build the code-aware retrieval channel

Build QAChat's code graph for `code_search`.

In [18]:
from qachat.retrievals import build_code_graph

code_graph = build_code_graph(source_files, source_units)

print(f"Code graph: {len(code_graph.entities)} entities, {len(code_graph.relations)} relations")


Code graph: 1886 entities, 1996 relations


## 9. Run retrieval

Define every channel's retrieval function against the real `corpus_chunks`
built earlier, and build each task's index once, reusing it below rather than
re-embedding the corpus per call. Real QAChat only ever calls
`run_hybrid_retrieval(..., with_code=True)` -- the rest
(`run_semantic_retrieval`, `run_lexical_retrieval`, `run_code_retrieval`, `run_hybrid_retrieval(..., with_code=False)`)
are ablations defined here to explain *why* the real pipeline scores what it
does.

Embedding is the slow part -- so each task's index is saved under the
project's own `indexes/` folder (`save_indexes`, the same mechanism
`RAGSession` itself uses) and reloaded next time if its fingerprint
(`indexes_are_current`) still matches the current chunks and code graph,
rebuilt otherwise.

In [19]:
import time

from tqdm.auto import tqdm

from benchmarks.common import format_query
from qachat import (
    build_lexical_index,
    build_semantic_index,
    code_search,
    hybrid_search,
    indexes_are_current,
    lexical_search,
    load_indexes,
    prepare_persistent_indexes,
    save_indexes,
    semantic_search,
)


def task_chunks(task: BenchmarkTask) -> list[Chunk]:
    """Return the real chunks belonging to one task's documents."""

    return [
        chunk
        for doc_id in task.corpus
        for chunk in chunks_by_doc_id.get(doc_id, ())
    ]


def run_semantic_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    semantic_index,
    embedding_session: EmbeddingSession,
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's semantic search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(
        task.queries.items(), desc=f"{task.name}: semantic retrieval", unit="query", leave=False
    )
    for query_id, query_text in query_items:
        ranked = semantic_search(
            semantic_index,
            format_query(query_text),
            embedding_session,
            count=count,
        )
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_lexical_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    lexical_index,
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's lexical search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(
        task.queries.items(), desc=f"{task.name}: lexical retrieval", unit="query", leave=False
    )
    for query_id, query_text in query_items:
        ranked = lexical_search(lexical_index, query_text, count=count)
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_code_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    *,
    count: int = 100,
) -> dict[str, dict[str, float]]:
    """Rank every query's real chunks with QAChat's code-aware search."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    query_items = tqdm(
        task.queries.items(), desc=f"{task.name}: code retrieval", unit="query", leave=False
    )
    for query_id, query_text in query_items:
        ranked = code_search(code_graph, chunks, query_text, count=count)
        results[query_id] = {
            result.chunk.identifier: result.score for result in ranked
        }
    return results


def run_hybrid_retrieval(
    task: BenchmarkTask,
    chunks: list[Chunk],
    semantic_index,
    lexical_index,
    embedding_session: EmbeddingSession,
    *,
    count: int = 100,
    with_code: bool = False,
) -> dict[str, dict[str, float]]:
    """Fuse semantic, lexical, and (optionally) code rankings."""

    count = min(count, len(chunks))
    results: dict[str, dict[str, float]] = {}
    channel_label = "hybrid+code" if with_code else "hybrid"
    query_items = tqdm(
        task.queries.items(), desc=f"{task.name}: {channel_label} retrieval", unit="query", leave=False
    )
    for query_id, query_text in query_items:
        semantic_ranked = semantic_search(
            semantic_index,
            format_query(query_text),
            embedding_session,
            count=count,
        )
        lexical_ranked = lexical_search(
            lexical_index, query_text, count=count
        )
        code_ranked = (
            code_search(code_graph, chunks, query_text, count=count)
            if with_code
            else ()
        )
        fused = hybrid_search(
            semantic=semantic_ranked,
            lexical=lexical_ranked,
            code=code_ranked,
            count=count,
        )
        results[query_id] = {
            result.chunk.identifier: result.score for result in fused
        }
    return results


task_indexes: dict[str, tuple[list[Chunk], object, object]] = {}
for task in tasks:
    chunks = task_chunks(task)
    index_path = project.indexes / f"{task.name}_retrieval.json"

    cached = None
    if index_path.exists():
        try:
            cached = load_indexes(index_path)
        except ValueError:
            cached = None
        if cached is not None and not indexes_are_current(cached, chunks, code_graph):
            cached = None

    if cached is not None:
        semantic_index = cached.semantic
        lexical_index = cached.lexical
        print(f"{task.name}: loaded cached index ({len(chunks)} chunks) from {index_path.relative_to(root)}")
    else:
        with tqdm(total=len(chunks), desc=f"Embedding {task.name} chunks") as progress_bar:

            def _on_embed_progress(current: int, total: int) -> None:
                progress_bar.n = current
                progress_bar.refresh()

            semantic_index = build_semantic_index(
                chunks, embedding_session, progress_callback=_on_embed_progress
            )

        _lexical_start = time.monotonic()
        lexical_index = build_lexical_index(chunks)
        print(f"{task.name}: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")

        save_indexes(
            prepare_persistent_indexes(semantic_index, lexical_index, code_graph),
            index_path,
        )
        print(f"{task.name}: indexed {len(chunks)} chunks, saved to {index_path.relative_to(root)}")

    task_indexes[task.name] = (chunks, semantic_index, lexical_index)


Embedding humaneval chunks: 100%|██████████| 196/196 [00:12<00:00, 15.79it/s]


humaneval: lexical index built in 0.0s.
humaneval: indexed 196 chunks, saved to benchmarks/sandbox/humaneval_mbpp_benchmark/indexes/humaneval_retrieval.json


Embedding mbpp chunks: 100%|██████████| 1027/1027 [00:47<00:00, 21.65it/s]


mbpp: lexical index built in 0.0s.
mbpp: indexed 1027 chunks, saved to benchmarks/sandbox/humaneval_mbpp_benchmark/indexes/mbpp_retrieval.json


## 10. Score the real pipeline: hybrid+code

`evaluate_retrieval` reimplements BEIR's `EvaluateRetrieval.evaluate`
and its custom MRR metric directly against `pytrec_eval`. Scored first:
`run_hybrid_retrieval(..., with_code=True)` -- the one call that
actually matches `RAGSession._retrieve`, since QAChat's real pipeline
never runs semantic search (or any single channel) on its own.
Everything else is an ablation.

In [20]:
from benchmarks.common import evaluate_retrieval

channel_scores: dict[str, dict[str, dict[str, dict[str, float]]]] = {
    "hybrid+code": {},
    "hybrid": {},
    "semantic": {},
    "lexical": {},
    "code": {},
}
task_results: dict[str, dict[str, dict[str, dict[str, float]]]] = {
    task.name: {} for task in tasks
}

for task in tasks:
    chunks, semantic_index, lexical_index = task_indexes[task.name]

    hybrid_code_results = run_hybrid_retrieval(
        task, chunks, semantic_index, lexical_index, embedding_session, with_code=True
    )
    channel_scores["hybrid+code"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], hybrid_code_results
    )
    task_results[task.name]["hybrid+code"] = hybrid_code_results

    headline = channel_scores["hybrid+code"][task.name]
    print(
        f"{task.name}: NDCG@10={headline['ndcg']['NDCG@10']}, "
        f"MRR@10={headline['mrr']['MRR@10']}, "
        f"Recall@10={headline['recall']['Recall@10']}"
    )

humaneval: NDCG@10=0.99303, MRR@10=0.99695, Recall@10=0.9939


mbpp: NDCG@10=0.66361, MRR@10=0.84073, Recall@10=0.70323


## 11. Compare: whole-document baseline vs. QAChat chunking

Scores the retrieval over `whole_document` chunks (CodeRAG-Bench's own un-split
granularity, comparable to the paper) and over QAChat's real chunking (`qachat_chunking`).

In [21]:
HEADLINE_MEASURES = (
    ("mrr", "MRR@10"),
    ("ndcg", "NDCG@10"),
    ("recall", "Recall@10"),
    ("recall", "Recall@100"),
)

whole_document_indexes: dict[str, tuple[list[Chunk], object, object]] = {}
for task in tasks:
    chunks = whole_document_chunks[task.name]
    with tqdm(total=len(chunks), desc=f"Embedding {task.name} whole documents") as progress_bar:

        def _on_embed_progress(current: int, total: int) -> None:
            progress_bar.n = current
            progress_bar.refresh()

        semantic_index = build_semantic_index(
            chunks, embedding_session, progress_callback=_on_embed_progress
        )
    _lexical_start = time.monotonic()
    lexical_index = build_lexical_index(chunks)
    print(f"{task.name}: lexical index built in {time.monotonic() - _lexical_start:.1f}s.")
    whole_document_indexes[task.name] = (chunks, semantic_index, lexical_index)
    print(f"{task.name}: indexed {len(chunks)} whole-document chunks")

comparison_scores: dict[str, dict[str, dict]] = {
    "whole_document": {},
    "qachat_chunking": {},
}

for task in tasks:
    wd_chunks, wd_semantic_index, wd_lexical_index = whole_document_indexes[task.name]
    wd_results = run_hybrid_retrieval(
        task, wd_chunks, wd_semantic_index, wd_lexical_index, embedding_session, with_code=True
    )
    comparison_scores["whole_document"][task.name] = evaluate_retrieval(task.qrels, wd_results)

    comparison_scores["qachat_chunking"][task.name] = channel_scores["hybrid+code"][task.name]

Embedding humaneval whole documents: 100%|██████████| 164/164 [00:15<00:00, 10.64it/s]


humaneval: lexical index built in 0.0s.
humaneval: indexed 164 whole-document chunks


Embedding mbpp whole documents: 100%|██████████| 500/500 [00:38<00:00, 12.85it/s]


mbpp: lexical index built in 0.0s.
mbpp: indexed 500 whole-document chunks


In [22]:
header = ["task", "mode", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>12}" for h in header))

for task in tasks:
    for mode in ("whole_document", "qachat_chunking"):
        scores = comparison_scores[mode][task.name]
        values = [
            f"{scores[group][label]:.3f}" for group, label in HEADLINE_MEASURES
        ]
        row = [task.name, mode, *values]
        print(" | ".join(f"{cell:>12}" for cell in row))

        task |         mode |       MRR@10 |      NDCG@10 |    Recall@10 |   Recall@100
   humaneval | whole_document |        0.997 |        0.998 |        1.000 |        1.000
   humaneval | qachat_chunking |        0.997 |        0.993 |        0.994 |        1.000
        mbpp | whole_document |        0.748 |        0.814 |        1.000 |        1.000
        mbpp | qachat_chunking |        0.841 |        0.664 |        0.703 |        0.992


**Reading the two numbers.** On HumanEval the two modes are almost identical -- only 31/164 documents split at all, so whole-document and QAChat chunking are nearly the same retrieval problem.

On MBPP they diverge: `whole_document` scores higher on NDCG@10 and Recall@10 (each document is one indivisible target, so finding it once is enough), while `qachat_chunking` scores higher on MRR@10 (splitting the document also splits its comment from its code, so there are now two relevant chunks per query instead of one -- MRR only needs the earliest of them to rank well, and the near-duplicate-text comment chunk is an easy early hit). NDCG@10 and Recall@10 drop under `qachat_chunking` because those metrics require *all* relevant chunks to rank well, not just the easiest one, and the code chunk alone is harder to retrieve from a natural-language query.

So `whole_document` is the fair, paper-comparable number; `qachat_chunking` is the honest number for QAChat's real pipeline, and the gap between them is a direct consequence of splitting one relevant unit into several.

For reference, CodeRAG-Bench's own reported numbers (Table 3, NDCG@10) for these two tasks, across every retriever they benchmarked:

| Retriever | HumanEval | MBPP |
|---|---|---|
| BM25 (lexical) | 100.0 | 98.6 |
| GIST-base (768) | 98.0 | 98.0 |
| GIST-large (1024) | 100.0 | 98.9 |
| BGE-base (768) | 99.7 | 98.0 |
| BGE-large (1024) | 98.0 | 99.0 |
| SFR-Mistral (4096) | 100.0 | 99.0 |
| Codesage-small (768) | 100.0 | 96.3 |
| Jina-v2-code (768) | 100.0 | 97.7 |
| OpenAI-03 (1536) | 100.0 | 98.9 |
| Voyage-code (1536) | 100.0 | 99.0 |

`whole_document`'s NDCG@10 above is the number directly comparable to this table.


## 12. Ablations: what's each piece contributing

Scores `semantic`, `lexical`, `code` alone, and `hybrid` without the code 
channel to show which channel is actually driving `hybrid+code`'s score.

In [23]:
for task in tasks:
    chunks, semantic_index, lexical_index = task_indexes[task.name]

    semantic_results = run_semantic_retrieval(task, chunks, semantic_index, embedding_session)
    channel_scores["semantic"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], semantic_results
    )
    task_results[task.name]["semantic"] = semantic_results

    lexical_results = run_lexical_retrieval(task, chunks, lexical_index)
    channel_scores["lexical"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], lexical_results
    )
    task_results[task.name]["lexical"] = lexical_results

    code_results = run_code_retrieval(task, chunks)
    channel_scores["code"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], code_results
    )
    task_results[task.name]["code"] = code_results

    hybrid_results = run_hybrid_retrieval(
        task, chunks, semantic_index, lexical_index, embedding_session, with_code=False
    )
    channel_scores["hybrid"][task.name] = evaluate_retrieval(
        qrels_chunks[task.name], hybrid_results
    )
    task_results[task.name]["hybrid"] = hybrid_results

In [24]:
header = ["task", "channel", *(label for _, label in HEADLINE_MEASURES)]
print(" | ".join(f"{h:>12}" for h in header))

for task in tasks:
    for channel in ("hybrid+code", "hybrid", "semantic", "lexical", "code"):
        scores = channel_scores[channel][task.name]
        values = [
            f"{scores[group][label]:.3f}" for group, label in HEADLINE_MEASURES
        ]
        row = [task.name, channel, *values]
        print(" | ".join(f"{cell:>12}" for cell in row))

        task |      channel |       MRR@10 |      NDCG@10 |    Recall@10 |   Recall@100
   humaneval |  hybrid+code |        0.997 |        0.993 |        0.994 |        1.000
   humaneval |       hybrid |        0.997 |        0.967 |        0.970 |        1.000
   humaneval |     semantic |        1.000 |        0.954 |        0.948 |        1.000
   humaneval |      lexical |        0.997 |        0.971 |        0.979 |        1.000
   humaneval |         code |        0.909 |        0.911 |        0.924 |        0.924
        mbpp |  hybrid+code |        0.841 |        0.664 |        0.703 |        0.992
        mbpp |       hybrid |        0.999 |        0.698 |        0.663 |        0.992
        mbpp |     semantic |        0.999 |        0.942 |        0.974 |        0.995
        mbpp |      lexical |        0.999 |        0.637 |        0.555 |        0.707
        mbpp |         code |        0.177 |        0.176 |        0.186 |        0.186


## 13. Results

**What the ablation shows.** MBPP's corpus documents split far more often than HumanEval's (500/500 vs. 31/164) — not because the code differs, but because `extract_source_units` always carves MBPP's #-comment task description into its own chunk, separate from the code. That gives most MBPP queries two gold chunks instead of one: a comment chunk that's near-verbatim the query, and a code chunk that isn't — which is why MRR@10 stays high (the comment chunk is easy to rank first) while Recall@10 (finding every gold chunk) drops for anything but semantic search.

The code-aware channel scores far better on HumanEval (queries contain the function name) than MBPP (queries are independent descriptions). But its effect on the fused result is asymmetric — helpful on HumanEval, actively harmful on MBPP, where a weak channel drags fusion down rather than just failing to help it.

## 14. Save results

In [25]:
import json
from benchmarks.common import local_run_timestamp

results_dir = root / "benchmarks" / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": "CodeRAG-bench (canonical: humaneval, mbpp) -- QAChat-native pipeline",
    "timestamp": timestamp,
    "embedding_model": MODEL_ID,
    "backend": selected_backend,
    "chunking_profile": {
        "target_tokens": profile.target_tokens,
        "max_tokens": profile.max_tokens,
        "overlap_tokens": profile.overlap_tokens,
    },
    "documents_split_into_multiple_chunks": split_counts,
    "code_graph_entities": len(code_graph.entities),
    "code_graph_relations": len(code_graph.relations),
    "scores": channel_scores,
    "comparison_scores": comparison_scores,
}

output_path = results_dir / f"{stamp}_canonical_qachat_native.json"
output_path.write_text(json.dumps(run_record, indent=2))

print(f"Saved: {output_path.relative_to(root)}")

Saved: benchmarks/results/20260926T102016+0200_canonical_qachat_native.json


## 15. Release the model

In [26]:
embedding_session.close()

print("Embedding model released.")

Embedding model released.
